In [1]:
# %%

#------------------------------------------------ Begin_Librairie ----------------------------------------

import pandas as pd

from bs4 import BeautifulSoup

from time import sleep

from datetime import datetime

from pandas import ExcelWriter

from selenium import webdriver

from selenium.webdriver.common.by import By

import datetime

import os

import re

import pdfplumber

from selenium.webdriver.chrome.service import Service as ChromeService

import xml.etree.ElementTree as ET 




In [2]:
# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------
regulatorName = 'IT IVASS'

print(f"Running {regulatorName} Web Scraping Tool v.1.0")

now=datetime.datetime.now()

filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - moodys.com\\Desktop\\Regulator\\{regulatorName}" ## to comment for the local environment

#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

writer = ExcelWriter(filename)

tempfolder=os.path.join(scriptfolder, 'tempfolder') 



if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)
    
# %%


Running IT IVASS Web Scraping Tool v.1.0


In [3]:
# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder}

chromeOptions.add_experimental_option("prefs",prefs)

driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()



In [4]:
# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict


# Define a function to scroll to the bottom of the page

def scroll_to_bottom(driver):

    # Get scroll height

    last_height = driver.execute_script("return document.body.scrollHeight")

    while True:

        # Scroll down to the bottom

        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        # Wait to load the page

        sleep(3)
        # Calculate new scroll height and compare with last scroll height

        new_height = driver.execute_script("return document.body.scrollHeight")

        if new_height == last_height:

            break

        last_height = new_height

def click_element_by_xpath(driver, xpath):

    # Find the element and click

    element = driver.find_element(By.XPATH, xpath)

    element.click()
    

def click_on_cookies(web_driver):

    try:

        web_driver.find_element(By.XPATH,f'//*[@id="modal-content-id-1"]/footer/div/button[3]').click()
        print('[Success] : Success to Click Cookie')

    except Exception as err:

        print('[ERROR] : Failed to click "I Accept" button on the cookies banner:', err)
        

def scrollinAndClick(xpath,key_press=False):
    if len(xpath) != 0 :
        for times in range(60):
            try:
                driver.find_element(By.XPATH, xpath).click()
                sleep(1)
                break
            except:
                print(f"[ERROR] : trying {times+1}/10 to key press 'DOWN' (scrolling)")
                sleep(1)
                if key_press:                    
                    driver.find_element(By.TAG_NAME, 'body').send_keys(key_press)
        else:   
            raise Exception(f'[ERROR] : Failed scrollin Or Click on xpath element : {xpath}')

def check_dowload_files(tempfolder, fileType, wait_time=10):

    for time in range(wait_time):

        if len([ele for ele in os.listdir(tempfolder) if '.crdownload' not in ele and '.tmp' not in ele]) != 0 :

            print(f"[INFO] : - {fileType} file = {os.listdir(tempfolder)[0]}")

            break

        else:

            print(f"[INFO] : - Download {fileType} file ... (wait {time*2}/{wait_time*2} s)")

            sleep(2)

    else:

        raise Exception(f'[ERROR] : - Failed to Download {fileType} file. Run Script again' )


In [5]:
# %%

#------------------------------------------------ Begin_Variable ----------------------------------------

regdict={

        'IT IVASS 1': 'https://infostat-ivass.bancaditalia.it/RIGAInquiry-public/ng/#/int-albi/search',
        # 'IT IVASS 2': 'https://infostat-ivass.bancaditalia.it/RIGAInquiry-public/ng/#/int-albi/search',
        # 'IT IVASS 3': 'https://infostat-ivass.bancaditalia.it/RIGAInquiry-public/ng/#/int-albi/search',
        # 'IT IVASS 4': 'https://infostat-ivass.bancaditalia.it/RIGAInquiry-public/ng/#/int-albi/search',
        # 'IT IVASS 5': 'https://infostat-ivass.bancaditalia.it/RIGAInquiry-public/ng/#/int-albi/search',
        # 'IT IVASS 6': 'https://infostat-ivass.bancaditalia.it/RIGAInquiry-public/ng/#/int-albi/search',
        # 'IT IVASS 7': 'https://infostat-ivass.bancaditalia.it/RIGAInquiry-public/ng/#/int-albi/search',
        # 'IT IVASS 8': 'https://infostat-ivass.bancaditalia.it/RIGAInquiry-public/ng/#/int-albi/search',

        }



Typology={

'IT IVASS 1':   'List I - insurance undertakings with registered office in another Member State admitted to operate in Italy under the establishment regime',
'IT IVASS 2':	'List II - insurance undertakings having their registered office in another Member State admitted to operate in Italy under the freedom to provide services',
'IT IVASS 3':	'List III - reinsurance undertakings having their registered office in another Member State admitted to operate in Italy under the establishment regime',
'IT IVASS 4':	'Section I - Insurance undertakings with registered office in Italy',
'IT IVASS 5':	'Section II - secondary establishments, established in Italy, of insurance undertakings with registered office in a third country',
'IT IVASS 6':	'Section III - special mutual insurance companies with registered office in Italy',
'IT IVASS 7':	'Section IV - reinsurance undertakings with registered office in Italy',
'IT IVASS 8':	'Section V - secondary establishments, established in Italy, of reinsurance undertakings with registered office in a third country',

        }



sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 

          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 

          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 

          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [],
        }


now = datetime.datetime.now()

processdate = now.strftime('%Y-%m-%d')



In [6]:

for reg in regdict:

    print(f'Working with list {reg}')

    driver.get(regdict[reg])

    sleep(5)

    language_button = driver.find_element(By.XPATH, '//*[@id="bs-example-navbar-collapse-1"]/ul/li[2]/div')

    language_button.click()

    sleep(3)

    en_option = driver.find_element(By.XPATH, '//*[@id="dropdown-alignment"]/li[2]')

    en_option.click()

    print('Choose English as Language')

    sleep(3)

    search_button = driver.find_element(By.XPATH, "//button[@type='submit']")

    search_button.click()

    print('Click  the Search Button')

    sleep(3)

    export_button = driver.find_element(By.XPATH, '//*[@id="button-export"]')

    export_button.click()

    csv_option = driver.find_element(By.XPATH, '//*[@id="dropdown-export"]/li[2]')

    csv_option.click()

    print('Download the csv file')

    


    # Read the CSV file into a list of lists
    sleep(5)
    
    
    
    file_path = os.path.join(tempfolder, os.listdir(tempfolder)[0])

    dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]

    with open(dl_files[0], 'r', encoding='latin1') as f:

        lines = f.readlines()

    data = [line.strip().split(';') for line in lines]

    max_columns = max(len(row) for row in data)



    # Ensure each row has the same number of columns by adding empty strings

    for row in data:

        while len(row) < max_columns:

            row.append('')



    # Convert the list of lists back into a DataFrame

    df = pd.DataFrame(data[1:],columns=data[0])

    df_cleaned = df[df['DENOMINAZIONE'] != '']

    

    sqldict = bourange_same_length_array(sqldict)

    os.chdir(scriptfolder)

    df=pd.DataFrame(sqldict)

    

    

    df['Name'] = df_cleaned['DENOMINAZIONE']

    df['InternalID_1'] = df_cleaned['NUMERO ISCRIZIONE']

    df['InternalID_1_type'] = 'Registration number'

    df['InternalID_1'] = df_cleaned['NUMERO ISCRIZIONE']

    df['InternalID_1_type'] = 'Registration number'

    df['InternalID_2_type'] = 'IVASS Code'

    df['InternalID_2'] = df_cleaned['CODICE IVASS']

    df['Cntry'] = df_cleaned['STATO']

    df['Address_1'] = df_cleaned['INDIRIZZO DG']

    df['Address_1'] = df['Address_1'].str.strip('"')

    df['RegulationDate'] = df_cleaned['DATA ISCRIZIONE']

    df['ListName'] = df_cleaned['CLASSIFICAZIONE']



    df['ListCode'] = df.apply(lambda row: '1' if row['ListName'] == 'IMPRESA SEE CHE OPERA IN STABILIMENTO' else

                                        '2' if row['ListName'] == 'IMPRESA SEE CHE OPERA IN LIBERA PRESTAZIONE DI SERVIZI' else

                                        '3' if row['ListName'] == 'IMPRESA DI RIASSICURAZIONE SEE CHE OPERA IN STABILIMENTO' else

                                        '4' if row['ListName'] == 'IMPRESA DI ASSICURAZIONE E RIASSICURAZIONE (LAVORO DIRETTO E INDIRETTO)' else

                                        '5' if row['ListName'] == 'RAPPRESENTANZA DI IMPRESA DI ASSICURAZIONE DI STATI TERZI (LAVORO DIRETTO E/O INDIRETTO)' else

                                        '7' if row['ListName'] == 'IMPRESA DI RIASSICURAZIONE (SOLO LAVORO INDIRETTO)' else

                                        row['ListCode'], axis=1)





    df['ListName'] = df['ListName'].replace({

        'IMPRESA SEE CHE OPERA IN STABILIMENTO': 'List I - insurance undertakings with registered office in another Member State admitted to operate in Italy under the establishment regime',

        'IMPRESA SEE CHE OPERA IN LIBERA PRESTAZIONE DI SERVIZI': 'List II - insurance undertakings having their registered office in another Member State admitted to operate in Italy under the freedom to provide services',

        'IMPRESA DI RIASSICURAZIONE SEE CHE OPERA IN STABILIMENTO': 'List III - reinsurance undertakings having their registered office in another Member State admitted to operate in Italy under the establishment regime',

        'IMPRESA DI ASSICURAZIONE E RIASSICURAZIONE (LAVORO DIRETTO E INDIRETTO)': 'Section I - Insurance undertakings with registered office in Italy',

        'RAPPRESENTANZA DI IMPRESA DI ASSICURAZIONE DI STATI TERZI (LAVORO DIRETTO E/O INDIRETTO)':'Section II - secondary establishments, established in Italy, of insurance undertakings with registered office in a third country',

        'IMPRESA DI RIASSICURAZIONE (SOLO LAVORO INDIRETTO)':'Section IV - reinsurance undertakings with registered office in Italy'

    })





    df['RegCtry']=reg.split(' ')[0]

    df['RegCode']=reg.split(' ')[1]

    df['RegulationType']='Regulated'

    df['ListProcessDate'] = processdate



    df = df.fillna('')

#df.to_csv('total.csv')

Working with list IT IVASS 1
Choose English as Language
Click  the Search Button
Download the csv file


In [7]:
# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------


df.to_excel(writer, 'SQL Ready', index=False)

writer.save()

writer.close()

driver.quit()

sleep(3)
    


C:\Users\wuj1\AppData\Local\Temp\2\ipykernel_17788\2021841299.py:6: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(writer, 'SQL Ready', index=False)


AttributeError: 'OpenpyxlWriter' object has no attribute 'save'

In [8]:
df

,bvdid,priority,ListLabel,Typology,EntryType,Name,InternalID_1,InternalID_1_type,InternalID_2,InternalID_2_type,...,RegulationTypeCode,RegulationDate,CancellationDate,RegCtry,RegCode,ListCode,ListLanguage,ListValidityDate,ListName,ListProcessDate
0,,,,,,ASSICURAZIONI GENERALI SOCIETA' PER AZIONI,1.00003,Registration number,A014S,IVASS Code,...,,2008-01-03,,IT,IVASS,4,,,Section I - Insurance undertakings with regist...,2024-11-15
1,,,,,,AXA ASSICURAZIONI S.P.A.,1.00025,Registration number,A037S,IVASS Code,...,,2008-01-03,,IT,IVASS,4,,,Section I - Insurance undertakings with regist...,2024-11-15
2,,,,,,GENERALI ITALIA S.P.A.,1.00021,Registration number,A044S,IVASS Code,...,,2008-01-03,,IT,IVASS,4,,,Section I - Insurance undertakings with regist...,2024-11-15
3,,,,,,HDI ASSICURAZIONI S.P.A.,1.00022,Registration number,A055X,IVASS Code,...,,2008-01-03,,IT,IVASS,4,,,Section I - Insurance undertakings with regist...,2024-11-15
4,,,,,,ITAS - ISTITUTO TRENTINO-ALTO ADIGE PER ASSICU...,1.00008,Registration number,A056M,IVASS Code,...,,2008-01-03,,IT,IVASS,4,,,Section I - Insurance undertakings with regist...,2024-11-15
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1087,,,,,,FORTEGRA BELGIUM INSURANCE COMPANY,II.01856,Registration number,41157,IVASS Code,...,,2024-04-23,,IT,IVASS,2,,,List II - insurance undertakings having their ...,2024-11-15
1088,,,,,,COMPAGNIE FRANÃAISE D'ASSURANCE POUR LE COMME...,II.01857,Registration number,41158,IVASS Code,...,,2024-06-10,,IT,IVASS,2,,,List II - insurance undertakings having their ...,2024-11-15
1089,,,,,,AIOI NISSAY DOWA LIFE INSURANCE OF EUROPE AKTI...,III.00013,Registration number,41159,IVASS Code,...,,2024-07-02,,IT,IVASS,2,,,List II - insurance undertakings having their ...,2024-11-15
1090,,,,,,"DIRECT POJISTOVNA, A.S.",II.01858,Registration number,41160,IVASS Code,...,,2024-08-30,,IT,IVASS,2,,,List II - insurance undertakings having their ...,2024-11-15


In [ ]:
df

NameError: name 'df' is not defined